# Implementing a Gated Recurrent Unit (GRU) Neural Network

## Setup

In [ ]:
import torch
import numpy as np

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

## Loading the data

In [ ]:
text = (
    "rosa canina rosa gallica rosa damascena rosa rugosa "
    "lavandula angustifolia lavandula stoechas lavandula latifolia "
    "tulipa gesneriana tulipa sylvestris tulipa clusiana "
    "orchis mascula orchis militaris orchis purpurea "
    "bellis perennis leucanthemum vulgare tanacetum parthenium "
    "helianthus annuus helianthus tuberosus helianthus debilis "
    "lilium candidum lilium martagon lilium bulbiferum "
    "jasminum officinale jasminum sambac jasminum nudiflorum "
    "iris germanica iris pseudacorus iris sibirica iris reticulata "
    "paeonia officinalis paeonia lactiflora paeonia suffruticosa "
    "magnolia grandiflora magnolia stellata magnolia soulangeana "
    "dahlia pinnata dahlia variabilis dahlia imperialis "
    "calendula officinalis tagetes erecta tagetes patula "
    "rhododendron ponticum rhododendron ferrugineum azalea japonica "
    "digitalis purpurea digitalis lanata digitalis lutea "
    "hydrangea macrophylla hydrangea paniculata hydrangea quercifolia "
    "syringa vulgaris syringa meyeri syringa microphylla "
    "hibiscus rosa-sinensis hibiscus syriacus hibiscus sabdariffa "
    "delphinium elatum delphinium grandiflorum delphinium consolida "
    "lupinus polyphyllus lupinus albus lupinus angustifolius "
    "salvia officinalis salvia splendens salvia nemorosa salvia rosmarinus "
    "echinacea purpurea echinacea angustifolia echinacea pallida "
    "narcissus pseudonarcissus narcissus poeticus narcissus jonquilla "
    "crocus sativus crocus vernus crocus chrysanthus crocus tommasinianus "
    "cyclamen persicum cyclamen hederifolium cyclamen coum "
    "freesia refracta freesia alba freesia corymbosa "
    "gardenia jasminoides gardenia thunbergia gardenia augusta "
    "gladiolus communis gladiolus italicus gladiolus palustris "
    "hyacinthus orientalis hyacinthus amethystinus hyacinthus romanus "
    "ranunculus acris ranunculus repens ranunculus bulbosus "
    "anemone coronaria anemone nemorosa anemone sylvestris "
    "clematis vitalba clematis armandii clematis montana "
    "wisteria sinensis wisteria floribunda wisteria frutescens "
    "aloe vera aloe arborescens aloe ferox aloe barbadensis "
    "monstera deliciosa monstera adansonii monstera obliqua "
    "philodendron hederaceum philodendron bipinnatifidum "
    "calathea orbifolia calathea makoyana calathea zebrina "
    "dracaena marginata dracaena fragrans dracaena sanderiana "
    "ficus benjamina ficus lyrata ficus elastica ficus pumila "
    "rosmarinus officinalis thymus vulgaris ocimum basilicum "
    "mentha piperita mentha spicata mentha aquatica "
    "origanum vulgare petroselinum crispum coriandrum sativum "
    "anethum graveolens allium schoenoprasum foeniculum vulgare "
    "matricaria chamomilla valeriana officinalis nepeta cataria "
    "quercus robur quercus petraea quercus ilex quercus suber "
    "acer platanoides acer campestre acer pseudoplatanus "
    "betula pendula betula pubescens betula papyrifera "
    "salix alba salix babylonica salix caprea salix viminalis "
    "prunus avium prunus cerasifera prunus serrulata "
    "rosa canina lavandula angustifolia tulipa gesneriana "
    "iris germanica paeonia lactiflora lilium candidum "
    "helianthus annuus magnolia grandiflora dahlia pinnata "
    "echinacea purpurea salvia officinalis lupinus polyphyllus "
)


chars = sorted(set(text))
vocab = {ch: i for i, ch in enumerate(chars)}
inv_vocab = {i: ch for ch, i in vocab.items()}
vocab_size = len(chars)

## Utility functions

In [ ]:
def encode(s: str) -> list[int]:
    return [vocab[c] for c in s]

def decode(ids: list[int]) -> str:
    return "".join(inv_vocab[i] for i in ids)

def one_hot(idx: int, size: int) -> torch.Tensor:
    vec = torch.zeros(size)
    vec[idx] = 1.0
    return vec

In [ ]:
def xavier(rows: int, cols: int) -> torch.Tensor:
    limit = np.sqrt(6.0 / (rows + cols))
    arr = np.random.uniform(-limit, limit, (rows, cols)).astype(np.float32)
    return torch.tensor(arr, requires_grad=True)

def zeros_param(*shape) -> torch.Tensor:
    return torch.zeros(*shape, requires_grad=True)

In [ ]:
def softmax(logits: torch.Tensor) -> torch.Tensor:
    e = torch.exp(logits - logits.max())
    return e / e.sum()

def cross_entropy(probs: torch.Tensor, target_idx: int) -> torch.Tensor:
    return -torch.log(probs[target_idx] + 1e-12)

def sgd_step(params: list[torch.Tensor], lr: float, clip: float = 5.0) -> None:
    with torch.no_grad():
        for p in params:
            if p.grad is None:
                continue
            p.grad.clamp_(-clip, clip)
            p -= lr * p.grad
            p.grad.zero_()

## The GRU

### Setting the hyperparameters

In [ ]:
hidden_size = 64
seq_len = 25
lr = 1e-2
epochs = 500

### Initialising weights and biases
This part differs from the LSTM.

In [ ]:
# Weights
W = xavier(vocab_size, 3 * hidden_size)
U_rz = xavier(hidden_size, 2 * hidden_size)
U_n = xavier(hidden_size, hidden_size)
V = xavier(hidden_size, vocab_size)

# Biases
b = zeros_param(3 * hidden_size)
b_h = zeros_param(vocab_size)

params = [W, U_rz, U_n, b, V, b_h]

### Implementing the GRU
This part differs from the LSTM.

In [ ]:
# This is the RNN core
def gru_step(
    x: torch.Tensor,
    h: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    rz_pre = x @ W[:, : 2 * hidden_size] + h @ U_rz + b[: 2 * hidden_size]
    z = torch.sigmoid(rz_pre[hidden_size:])
    r = torch.sigmoid(rz_pre[:hidden_size])

    n = torch.tanh(x @ W[:, 2 * hidden_size :] + (r * h) @ U_n + b[2 * hidden_size :])

    h_new = ... # TODO: Add code here
    
    logits = ... # TODO: Add code here
    return h_new, logits

def gru_forward(
    inputs: list[int],
    targets: list[int],
    h0: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    h = h0
    loss = torch.tensor(0.0)

    for t, (x_idx, y_idx) in enumerate(zip(inputs, targets)):
        x = one_hot(x_idx, vocab_size)
        h, logits = gru_step(x, h)
        probs = softmax(logits)
        loss = loss + cross_entropy(probs, y_idx)

    return loss / len(inputs), h

### Training the GRU


In [ ]:
data = encode(text)
n_data = len(data)

print("Starting training ...")
for epoch in range(1, epochs + 1):
    total_loss = 0.0
    n_chunks = 0
    h_state = torch.zeros(hidden_size)

    for start in range(0, n_data - seq_len - 1, seq_len):
        inputs = data[start : start + seq_len]
        targets = data[start + 1 : start + seq_len + 1]

        h_prev = h_state.detach()
        loss, h_state = gru_forward(inputs, targets, h_prev)

        loss.backward()
        sgd_step(params, lr)

        total_loss += loss.item()
        n_chunks += 1

    if epoch % 50 == 0:
        avg_loss = total_loss / max(n_chunks, 1)
        print(f"Epoch {epoch:4d}/{epochs}, average loss = {avg_loss:.4f}")


## Sampling and testing

In [ ]:
def sample(seed_char: str, length: int = 200, temperature: float = 0.8) -> str:
    h = torch.zeros(hidden_size)
    idx = vocab[seed_char]
    out = [seed_char]

    with torch.no_grad():
        for _ in range(length):
            x = one_hot(idx, vocab_size)
            h, logits = gru_step(x, h)
            probs = softmax(logits / temperature)
            idx = int(np.random.choice(vocab_size, p=probs.numpy()))
            out.append(inv_vocab[idx])

    return "".join(out)

In [ ]:
print(sample("r", length=400, temperature=0.8))